# tiny transformer from scratch

building the smallest plausible decoder-only transformer in pytorch. tokens are characters of a tiny shakespeare-ish string. just to see the shapes line up. proper version will be its own repo in 2022.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class MHSA(nn.Module):
    def __init__(self, d, n_heads):
        super().__init__()
        self.qkv = nn.Linear(d, 3*d)
        self.proj = nn.Linear(d, d)
        self.n_heads = n_heads
    def forward(self, x, mask):
        B, T, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        H = self.n_heads
        q = q.view(B, T, H, D//H).transpose(1, 2)
        k = k.view(B, T, H, D//H).transpose(1, 2)
        v = v.view(B, T, H, D//H).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(D//H)
        att = att.masked_fill(mask == 0, -1e9)
        att = F.softmax(att, dim=-1)
        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, D)
        return self.proj(y)

## block + model

In [ ]:
class Block(nn.Module):
    def __init__(self, d, n_heads):
        super().__init__()
        self.ln1 = nn.LayerNorm(d)
        self.ln2 = nn.LayerNorm(d)
        self.attn = MHSA(d, n_heads)
        self.mlp = nn.Sequential(nn.Linear(d, 4*d), nn.GELU(), nn.Linear(4*d, d))
    def forward(self, x, mask):
        x = x + self.attn(self.ln1(x), mask)
        x = x + self.mlp(self.ln2(x))
        return x

class TinyTfmr(nn.Module):
    def __init__(self, vocab, d=64, n_heads=4, n_layers=2, ctx=32):
        super().__init__()
        self.tok = nn.Embedding(vocab, d)
        self.pos = nn.Embedding(ctx, d)
        self.blocks = nn.ModuleList([Block(d, n_heads) for _ in range(n_layers)])
        self.ln = nn.LayerNorm(d)
        self.head = nn.Linear(d, vocab)
        self.ctx = ctx
        mask = torch.tril(torch.ones(ctx, ctx)).view(1, 1, ctx, ctx)
        self.register_buffer('mask', mask)
    def forward(self, idx):
        B, T = idx.shape
        x = self.tok(idx) + self.pos(torch.arange(T, device=idx.device))
        for blk in self.blocks:
            x = blk(x, self.mask[:, :, :T, :T])
        return self.head(self.ln(x))

## tiny char-level training, just to verify gradients flow

In [ ]:
text = 'hello world ' * 200
chars = sorted(set(text))
stoi = {c:i for i,c in enumerate(chars)}
itos = {i:c for i,c in enumerate(chars)}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)

ctx = 16
model = TinyTfmr(len(chars), d=32, n_heads=4, n_layers=2, ctx=ctx)
opt = torch.optim.Adam(model.parameters(), lr=3e-4)

for step in range(200):
    ix = torch.randint(0, len(data) - ctx - 1, (32,))
    x = torch.stack([data[i:i+ctx] for i in ix])
    y = torch.stack([data[i+1:i+ctx+1] for i in ix])
    logits = model(x)
    loss = F.cross_entropy(logits.view(-1, len(chars)), y.view(-1))
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 50 == 0:
        print(step, loss.item())

In [ ]:
# add weight decay
# opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)


added gradient clipping at 1.0, training stabilized.

label smoothing 0.1 gave a tiny lift.

moved seed init to the top, reproducibility check passed.

## obs
lr=3e-5 way better than 5e-5 here.

In [ ]:
# few more steps
STEPS = 200

In [ ]:
# use fp16 if available
dtype = torch.float16 if torch.cuda.is_available() else torch.float32

## note
this crashed on cpu without `device_map='auto'`. moving on.